In [ ]:
!nvidia-smi

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
%%capture
import os
if not os.path.exists('/content/drive/MyDrive/AAAI_TALAS/AAAI-TALAS-main'):
    !unzip /content/drive/MyDrive/AAAI_TALAS/AAAI-TALAS-main.zip

In [3]:
%cd /content/AAAI-TALAS-main

/content/AAAI-TALAS-main


In [5]:
# Remove stale cached teacher embeddings before training
!rm -rf cache
!mkdir -p cache
!echo "Removed cache/"

Removed cache/


In [ ]:
!pip install -r requirements.txt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.9/72.9 kB 5.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 133.4/133.4 kB 14.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
INFO: pip is looking at multiple versions of notebook to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 300.8/300.8 kB 28.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 153.3/153.3 kB 18.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.8/118.8 kB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.7/109.7 kB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 72.5 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 139.9 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.6/14.6 MB 140.1 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 134.3 MB/s et

In [6]:
!sed -i 's|\.\./main.py|main.py|g' scripts/train_talas.sh
!sed -i 's|TRAIN_DATA=.*|TRAIN_DATA="data/merged_9_data_3k_each_ver2.csv"|g' scripts/train_talas.sh
!sed -i 's|TEACHER_MODEL=.*|TEACHER_MODEL="Qwen/Qwen3-Embedding-4B"|g' scripts/train_talas.sh
!sed -i 's|STUDENT_MODEL=.*|STUDENT_MODEL="google-bert/bert-base-uncased"|g' scripts/train_talas.sh
!cat scripts/train_talas.sh | grep -E "TRAIN_DATA|TEACHER_MODEL|STUDENT_MODEL"

!sed -i -E 's/BATCH_SIZE=[0-9]+/BATCH_SIZE=256/g' scripts/train_talas.sh
!sed -i -E 's/(--batch_size\s+)[0-9]+/\1256/g' scripts/train_talas.sh
!sed -i -E 's/(--per_device_train_batch_size\s+)[0-9]+/\1256/g' scripts/train_talas.sh

# Kiểm tra lại các dòng có chứa từ khoá batch trong script
!cat scripts/train_talas.sh | grep -i "batch"
!test -f data/test_debug.csv || (echo 'Missing data/test_debug.csv. Re-run the unzip cell or check the archive.' && false)
!bash scripts/train_talas.sh

TRAIN_DATA="data/merged_9_data_3k_each_ver2.csv"
STUDENT_MODEL="google-bert/bert-base-uncased"
TEACHER_MODEL="Qwen/Qwen3-Embedding-4B"
    --train_data $TRAIN_DATA \
    --student_model $STUDENT_MODEL \
    --teacher_model $TEACHER_MODEL \
BATCH_SIZE=256
    --batch_size $BATCH_SIZE \
Training with TALAS method

Configuration for TALAS method:
  train_data_path           : data/merged_9_data_3k_each_ver2.csv
  student_model_name        : google-bert/bert-base-uncased
  teacher_model_name        : Qwen/Qwen3-Embedding-4B
  batch_size                : 256
  epochs                    : 5
  learning_rate             : 2e-05
  max_length                : 256
  save_dir                  : checkpoints/talas

Done setup_seed with seed=42
[WARN] Only 1 GPU available -> both on cuda:0
Done setup_devices
Loading tokenizers...
Loading student model: google-bert/bert-base-uncased
Loading weights: 100% 199/199 [00:00<00:00, 5465.84it/s]
[transformers] BertModel LOAD REPORT from: google-bert/bert-bas

In [7]:
# Run all validation and test benchmarks from the saved TALAS checkpoint
import re
from pathlib import Path

import torch
from transformers import AutoModel

from src.evaluation.evaluation_automodel import (
    eval_classification_task,
    eval_pair_task,
    eval_sts_task,
    eval_cls_tasks,
    eval_pair_tasks,
    eval_sts_tasks,
    test_cls_tasks,
    test_pair_tasks,
    test_sts_tasks,
)

checkpoint_dir = Path("checkpoints/talas")
best_checkpoint = checkpoint_dir / "best_model.pt"

if best_checkpoint.exists():
    checkpoint_path = best_checkpoint
else:
    checkpoints = sorted(
        checkpoint_dir.glob("checkpoint_epoch_*.pt"),
        key=lambda p: int(re.search(r"checkpoint_epoch_(\d+)", p.stem).group(1)),
    )
    assert checkpoints, f"No checkpoint found in {checkpoint_dir}"
    checkpoint_path = checkpoints[-1]

print(f"Loading checkpoint: {checkpoint_path}")

try:
    checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
except TypeError:
    checkpoint = torch.load(checkpoint_path, map_location="cpu")

cfg = checkpoint.get("config", {})
student_model_name = cfg.get("student_model_name", "google-bert/bert-base-uncased")
print(f"Student model: {student_model_name}")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

model = AutoModel.from_pretrained(student_model_name)
model.load_state_dict(checkpoint["model_state_dict"])
model.to(device)
model.eval()

benchmark_groups = [
    ("Validation classification", eval_classification_task, eval_cls_tasks),
    ("Validation pair classification", eval_pair_task, eval_pair_tasks),
    ("Validation STS", eval_sts_task, eval_sts_tasks),
    ("Test classification", eval_classification_task, test_cls_tasks),
    ("Test pair classification", eval_pair_task, test_pair_tasks),
    ("Test STS", eval_sts_task, test_sts_tasks),
]

for name, eval_fn, tasks in benchmark_groups:
    print("\n" + "=" * 80)
    print(name)
    print("=" * 80)
    eval_fn(model, tasks)

print("\nAll benchmark evaluations finished.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:122: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


Loading checkpoint: checkpoints/talas/best_model.pt
Student model: google-bert/bert-base-uncased
Using device: cuda


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: google-bert/bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Validation classification
 eval classifier
data/multi-data/banking77_validation.csv


...

Spearman: 0.7849
data/multi-data/sts12_test.csv


100%|██████████| 49/49 [00:01<00:00, 25.67it/s]


Spearman: 0.7275
data/multi-data/stsb_test.csv


100%|██████████| 22/22 [00:00<00:00, 25.10it/s]

Spearman: 0.7999

All benchmark evaluations finished.
